# 03. CDI 재설계 노트북
## 생활 인프라 사막 지수 (Community Desert Index) — v2

**재설계 배경**

기존 CDI는 `total_infra_count`에 단순 Min-Max 정규화를 적용했는데,  
역삼1동(772개) 하나가 extreme outlier로 작동해 나머지 대부분의 동(Q3=139개)이  
CDI ≈ 0.86~0.93에 밀집되는 **변별력 소실** 문제가 발생했습니다.

**재설계 핵심 원칙**
1. **고령자 기준 정규화**: `시설 수 / (고령인구 / 1000)` → 인구 규모 보정
2. **카테고리별 가중치**: 약국(×2.0) > 의료(×1.5) > 편의점(×1.0) > 식료품(×0.5)  
   — 고령 1인가구가 비가 오는 날 가장 필요한 시설 순서 반영
3. **로그 변환**: `log1p()` 적용으로 outlier 영향 완화
4. **역전**: 접근성이 낮을수록 CDI 높음 (1 − normalized)

**노인의료복지시설(2025) 데이터 제외 사유**
- 기준일 2024.12.31 → 2021년 분석과 4년 시간 불일치
- 행정동 매핑에 도로명주소 → 행정동 지오코딩 필요 (별도 추가 작업)
- CDI 변별력 문제는 정규화 방법에서 발생하므로 데이터 추가로 해결 불가
- 정책제안 섹션에서 현행 시설 분포 참고 자료로 별도 활용

---
## 1. 라이브러리 및 경로 설정

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE_DIR  = Path('c:/Tsum2026/T_SUM2026')
PROC_DIR  = BASE_DIR / 'data' / 'processed'
DIAG_DIR  = BASE_DIR / 'outputs' / 'tables'

CDI_ALIGNED_PATH = PROC_DIR / 'cdi_by_dong_2021_aligned.csv'   # 기존 CDI (원본 시설 수)
IVI_PATH          = PROC_DIR / 'ivi_social_isolation_2021.csv'  # 행정동별 elderly_population
OUT_CDI_V2        = PROC_DIR / 'cdi_by_dong_2021_v2.csv'        # 재설계 CDI 저장 경로

# 카테고리별 가중치 — 고령 1인가구의 비 오는 날 필수도 반영
INFRA_WEIGHTS = {
    'pharmacy_count':   2.0,   # 약국: 가장 필수
    'medical_count':    1.5,   # 의료시설
    'convenience_count': 1.0,  # 편의점
    'grocery_count':    0.5,   # 식료품점
}

for label, path in [
    ('기존 CDI aligned', CDI_ALIGNED_PATH),
    ('IVI (elderly 인구 포함)', IVI_PATH),
    ('전처리 저장 폴더', PROC_DIR),
]:
    mark = '✓' if path.exists() else '✗ 없음!'
    print(f'[{mark}] {label}: {path}')

---
## 2. 데이터 불러오기

In [ ]:
cdi_old = pd.read_csv(CDI_ALIGNED_PATH, encoding='utf-8-sig')
ivi      = pd.read_csv(IVI_PATH,         encoding='utf-8-sig')

print('기존 CDI shape:', cdi_old.shape)
print('IVI shape:     ', ivi.shape)
print()
print('기존 CDI 컬럼:', cdi_old.columns.tolist())
print('IVI 컬럼:     ', ivi.columns.tolist())

---
## 3. 기존 CDI 문제 진단

In [ ]:
print('=== 기존 CDI 분포 ===')  
print(cdi_old['CDI'].describe().round(4))
print()
print('=== total_infra_count 분포 ===')  
print(cdi_old['total_infra_count'].describe().round(1))
print()
print('상위 outlier 동 (infra 가장 많음):')
display(
    cdi_old.nlargest(5, 'total_infra_count')[
        ['자치구명','행정동명','total_infra_count','infra_score','CDI']
    ]
)
print()
print('★ 문제: 역삼1동(772개)이 Min-Max 기준값이 되어')
print('   중위수(97개)인 동의 infra_score = {:.3f} → CDI = {:.3f}'.format(
    (97 - cdi_old['total_infra_count'].min()) / (cdi_old['total_infra_count'].max() - cdi_old['total_infra_count'].min()),
    1 - (97 - cdi_old['total_infra_count'].min()) / (cdi_old['total_infra_count'].max() - cdi_old['total_infra_count'].min())
))
print('   → 대부분의 동이 CDI 0.86~0.93에 밀집, 변별력 소실')

---
## 4. elderly_population 병합

In [ ]:
# IVI에서 elderly_population, total_population 가져오기
ivi_pop = ivi[['gu', 'dong', 'elderly_population', 'total_population']].copy()
ivi_pop = ivi_pop.rename(columns={'gu': '자치구명', 'dong': '행정동명'})

df = cdi_old.merge(ivi_pop, on=['자치구명', '행정동명'], how='left')

miss_eld = df['elderly_population'].isna().sum()
print(f'elderly_population 병합 후 결측: {miss_eld}건')

if miss_eld > 0:
    miss_rows = df[df['elderly_population'].isna()]
    print('결측 행정동:')  
    print(miss_rows[['자치구명','행정동명']].to_string(index=False))
    # 결측은 전체 고령화율 평균(약 15%)으로 대체
    fallback_ratio = 0.15
    df.loc[df['elderly_population'].isna(), 'elderly_population'] = (
        df.loc[df['elderly_population'].isna(), 'total_population'].fillna(10000) * fallback_ratio
    )
    print(f'  → 평균 고령화율({fallback_ratio*100:.0f}%) 적용으로 대체')

print(f'\n최종 행 수: {len(df)}')
display(df[['자치구명','행정동명','elderly_population','total_population'] + list(INFRA_WEIGHTS.keys())].head())

---
## 5. 가중 시설 접근성 점수 산출

$$\text{weighted\_infra} = \text{pharmacy} \times 2 + \text{medical} \times 1.5 + \text{convenience} \times 1 + \text{grocery} \times 0.5$$

$$\text{infra\_per\_1000eld} = \frac{\text{weighted\_infra}}{\text{elderly\_population} / 1000}$$

$$\text{log\_infra} = \log(1 + \text{infra\_per\_1000eld})$$

$$\text{CDI} = 1 - \text{MinMax}(\text{log\_infra})$$

In [ ]:
# 1) 가중 시설 합산
df['weighted_infra'] = sum(
    df[col].fillna(0) * weight
    for col, weight in INFRA_WEIGHTS.items()
)

# 2) 고령자 1000명당 가중 시설 수
df['elderly_pop_safe'] = df['elderly_population'].clip(lower=1)  # 0 나누기 방지
df['infra_per_1000eld'] = df['weighted_infra'] / (df['elderly_pop_safe'] / 1000)

# 3) 로그 변환 (outlier 완화)
df['log_infra'] = np.log1p(df['infra_per_1000eld'])

# 4) Min-Max 정규화 (높을수록 접근성 좋음)
mn = df['log_infra'].min()
mx = df['log_infra'].max()
df['infra_accessibility_score'] = (df['log_infra'] - mn) / (mx - mn)

# 5) CDI = 1 - 접근성 (높을수록 사막)
df['CDI'] = 1 - df['infra_accessibility_score']

print('=== 재설계 CDI 분포 (v2) ===')  
print(df['CDI'].describe().round(4))
print()
print('★ 개선 비교:')
print(f'  기존: 평균 {cdi_old["CDI"].mean():.3f}, 표준편차 {cdi_old["CDI"].std():.3f}')
print(f'  신규: 평균 {df["CDI"].mean():.3f}, 표준편차 {df["CDI"].std():.3f}')

---
## 6. 순위 및 등급 산출

In [ ]:
# 순위 (CDI 높을수록 = 더 사막 = rank 1)
df['CDI_rank'] = df['CDI'].rank(ascending=False, method='min').astype(int)

# 등급 1(최저위험) ~ 5(최고위험)
df['CDI_grade'] = pd.qcut(
    df['CDI'].rank(method='first'),
    q=5,
    labels=[1, 2, 3, 4, 5],
    duplicates='drop'
).astype(int)

print('=== CDI_grade 분포 ===')  
print(df['CDI_grade'].value_counts().sort_index())
print()
print('=== CDI 상위 15개 (가장 사막인 동) ===')  
display(
    df.nsmallest(15, 'CDI_rank')[
        ['자치구명','행정동명','elderly_population','weighted_infra',
         'infra_per_1000eld','CDI','CDI_rank','CDI_grade']
    ].round(3)
)
print()
print('=== CDI 하위 10개 (인프라 가장 풍부한 동) ===')  
display(
    df.nlargest(10, 'CDI_rank')[
        ['자치구명','행정동명','elderly_population','weighted_infra',
         'infra_per_1000eld','CDI','CDI_rank','CDI_grade']
    ].round(3)
)

---
## 7. 기존 CDI와 비교 검증

In [ ]:
compare = df[['자치구명','행정동명','CDI']].copy()
compare = compare.merge(
    cdi_old[['자치구명','행정동명','CDI']].rename(columns={'CDI':'CDI_old'}),
    on=['자치구명','행정동명']
)

print('=== 기존 vs 신규 CDI 상관계수 ===')  
print(f'  Pearson r = {compare["CDI"].corr(compare["CDI_old"]):.3f}')
print()
print('=== 기존 CDI에서 고위험(grade=5)이었으나 신규에서 제외된 동 (상위 10개) ===')  
old_grade5 = cdi_old[cdi_old['CDI_grade']==5][['자치구명','행정동명']]
new_not5 = df[df['CDI_grade']!=5][['자치구명','행정동명']]
downgraded = old_grade5.merge(new_not5, on=['자치구명','행정동명'])
if len(downgraded) > 0:
    tmp = compare.merge(downgraded, on=['자치구명','행정동명'])
    display(tmp.nlargest(10, 'CDI_old')[['자치구명','행정동명','CDI_old','CDI']].round(3))
else:
    print('  없음')
print()
print('=== 신규 CDI에서 새롭게 고위험(grade=5)으로 진입한 동 (상위 10개) ===')  
new_grade5 = df[df['CDI_grade']==5][['자치구명','행정동명']]
old_not5 = cdi_old[cdi_old['CDI_grade']!=5][['자치구명','행정동명']]
upgraded = new_grade5.merge(old_not5, on=['자치구명','행정동명'])
if len(upgraded) > 0:
    tmp = compare.merge(upgraded, on=['자치구명','행정동명'])
    display(tmp.nlargest(10, 'CDI')[['자치구명','행정동명','CDI_old','CDI']].round(3))
else:
    print('  없음')

---
## 8. 최종 저장

In [ ]:
SAVE_COLS = [
    '기준연도', '자치구명', '행정동코드', '행정동명',
    'convenience_count', 'grocery_count', 'pharmacy_count', 'medical_count',
    'total_infra_count', 'elderly_population',
    'weighted_infra', 'infra_per_1000eld', 'log_infra',
    'infra_accessibility_score', 'CDI', 'CDI_rank', 'CDI_grade',
]
df_out = df[SAVE_COLS].copy()
df_out.to_csv(OUT_CDI_V2, index=False, encoding='utf-8-sig')
print(f'저장 완료: {OUT_CDI_V2}')
print(f'행 수: {len(df_out)}, 열 수: {len(df_out.columns)}')

# 검증
print()
print('=== 최종 검증 ===')
v = pd.read_csv(OUT_CDI_V2, encoding='utf-8-sig')
print(f'결측치: {v["CDI"].isna().sum()}건')
print(f'CDI 0~1 여부: {bool(v["CDI"].between(0,1).all())}')
print(f'CDI_grade 1~5 여부: {bool(v["CDI_grade"].between(1,5).all())}')
print(f'행정동코드 중복: {v["행정동코드"].duplicated().sum()}건')
print(f'CDI 평균: {v["CDI"].mean():.3f}, 표준편차: {v["CDI"].std():.3f}')
print('✓ 검증 완료')